In [1]:
import os
import numpy as np
import pandas as pd
import xarray as xr

# -------------------------
# PATHS
# -------------------------
base = os.path.expanduser(
    "~/Documents/summer 2025/python/bing/papers/biomass/Analysis"
)

nc_files = [
    os.path.join(base, "Ocean_Biogeochemistry_BGC-Argo_Global_Profiles_GulfofMexico.nc"),
    os.path.join(base, "Ocean_Biogeochemistry_BGC-Argo_Global_Profiles_Mediterranean.nc")
]

rows = []

# -------------------------
# LOOP THROUGH FILES
# -------------------------
for path in nc_files:

    print("\nProcessing:", os.path.basename(path))

    ds = xr.open_dataset(path)

    n_profiles = ds.dims["N_STATIONS"]

    for i in range(n_profiles):

        try:
            prof = ds.isel(N_STATIONS=i)

            # QC filter (same as PI)
            good = prof["Particle_backscattering_at_700_nm_adjusted__qc"].data <= 50

            if not np.any(good):
                continue

            # depth
            depth = prof["Pressure_adjusted_"].data[good]
            bbp = prof["Particle_backscattering_at_700_nm_adjusted_"].data[good]

            # require surface + MLD coverage (same logic)
            if np.sum(depth < 20) < 3:
                continue

            # -------------------------
            # compute top 20 m median
            # -------------------------
            mask = depth <= 20
            vals = bbp[mask]

            if len(vals) == 0:
                continue

            median_val = float(np.median(vals))

            # file name (WMOQC.nc style)
            wmo = str(prof["Platform_Number"].data.astype(str)).strip()
            profile_num = i

            filename = f"{wmo}QC.nc"

            rows.append([
                filename,
                median_val,
                len(vals)
            ])

        except Exception as e:
            print("Skipping", i, e)


# -------------------------
# CREATE OUTPUT
# -------------------------
df = pd.DataFrame(
    rows,
    columns=[
        "file",
        "bbp700_top20m_median",
        "n_values_used"
    ]
)

# -------------------------
# SAVE
# -------------------------
out_path = os.path.join(base, "ocean_biogeochem_top20m_bbp.csv")
df.to_csv(out_path, index=False)

print("\nSaved to:", out_path)
print(df.head())
print("Total rows:", len(df))


Processing: Ocean_Biogeochemistry_BGC-Argo_Global_Profiles_GulfofMexico.nc


/var/folders/54/5kc2y0693j37p_j3_04pyxkc0000gn/T/ipykernel_61506/638337514.py:29: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  n_profiles = ds.dims["N_STATIONS"]
/var/folders/54/5kc2y0693j37p_j3_04pyxkc0000gn/T/ipykernel_61506/638337514.py:29: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  n_profiles = ds.dims["N_STATIONS"]



Processing: Ocean_Biogeochemistry_BGC-Argo_Global_Profiles_Mediterranean.nc

Saved to: /Users/allie/Documents/summer 2025/python/bing/papers/biomass/Analysis/ocean_biogeochem_top20m_bbp.csv
           file  bbp700_top20m_median  n_values_used
0  7901009QC.nc              0.000644             10
1  7901009QC.nc              0.000537              9
2  4903622QC.nc              0.000572              3
3  7901009QC.nc              0.000559              9
4  4903622QC.nc              0.000561              3
Total rows: 266
